# Prerequisites: Set up MLflow and Optuna

In [1]:
pip install mlflow optuna

Note: you may need to restart the kernel to use updated packages.


# Step 1: Create a new experiment

In [2]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Hyperparameter Tuning Experiment")

2025/12/29 17:40:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 17:40:45 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 17:40:45 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 17:40:45 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 17:40:45 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 17:40:45 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/home/junspring/mlflow-jjb/mlruns/2', creation_time=1766993925693, experiment_id='2', last_update_time=1766993925693, lifecycle_stage='active', name='Hyperparameter Tuning Experiment', tags={}>

# Step 2: Prepare Your Data

In [3]:
import pandas as pd
import kagglehub
import os
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download("sooyoungher/california-housing")

csv_path = os.path.join(path, "fetch_california_housing.csv")
df = pd.read_csv(csv_path)

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

/home/junspring/anaconda3/envs/mlflow/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 3: Define the objective function

In [4]:
import mlflow
import optuna
import sklearn


def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        rf_max_depth = trial.suggest_int("rf_max_depth", 2, 32)
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=10)
        rf_max_features = trial.suggest_float("rf_max_features", 0.2, 1.0)
        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
            "max_features": rf_max_features,
        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = sklearn.ensemble.RandomForestRegressor(**params)
        regressor_obj.fit(X_train, y_train)

        y_pred = regressor_obj.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        # Log current trial's error metric
        mlflow.log_metrics({"error": error})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error

# Step 3: Run the hyperparameter tuning study

In [5]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2025-12-29 17:40:53,936] A new study created in memory with name: no-name-48361c53-1069-44a0-951f-850f4b816a48
[I 2025-12-29 17:41:01,704] Trial 0 finished with value: 0.24413359101491647 and parameters: {'rf_max_depth': 21, 'rf_n_estimators': 220, 'rf_max_features': 0.4052235171458666}. Best is trial 0 with value: 0.24413359101491647.
[I 2025-12-29 17:41:04,560] Trial 1 finished with value: 0.2840210217804818 and parameters: {'rf_max_depth': 12, 'rf_n_estimators': 50, 'rf_max_features': 0.7779750658980382}. Best is trial 0 with value: 0.24413359101491647.
[I 2025-12-29 17:41:06,403] Trial 2 finished with value: 0.6217144415846492 and parameters: {'rf_max_depth': 3, 'rf_n_estimators': 150, 'rf_max_features': 0.414386018683898}. Best is trial 0 with value: 0.24413359101491647.
[I 2025-12-29 17:41:15,681] Trial 3 finished with value: 0.2633926451212368 and parameters: {'rf_max_depth': 15, 'rf_n_estimators': 220, 'rf_max_features': 0.7308824647417747}. Best is trial 0 with value: 0.244

# Step 5: Register Your Best Model

In [6]:
# Register the best model using the model URI
mlflow.register_model(
    model_uri="runs:/14064e927c0d4c908646def200c043dc/model",
    name="housing-price-predictor",
)

# > Successfully registered model 'housing-price-predictor'.
# > Created version '1' of model 'housing-price-predictor'.

2025/12/29 17:54:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 17:54:12 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 17:54:12 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 17:54:12 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Registered model 'housing-price-predictor' already exists. Creating a new version of this model...
2025/12/29 17:54:12 WARNING mlflow.tracking._model_registry.fluent: Run with id 14064e927c0d4c908646def200c043dc has no artifacts at artifact path 'model', registering model based on models:/m-174732f844b7496088cd8ddf2b47aa45 instead
Created version '2' of model 'housing-price-predictor'.


<ModelVersion: aliases=[], creation_timestamp=1766998452071, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1766998452071, metrics=None, model_id=None, name='housing-price-predictor', params=None, run_id='14064e927c0d4c908646def200c043dc', run_link=None, source='models:/m-174732f844b7496088cd8ddf2b47aa45', status='READY', status_message=None, tags={}, user_id=None, version=2>